[◀ Notebook 10: Magic Methods](10_advanced_data_model.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 12: Iterators & Generators ▶](12_iterators_and_generators.ipynb)

# Notebook 11: Type Hinting and Static Analysis

Python is historically a dynamically typed language: types are resolved at runtime, and names can reference objects of any type during execution. However, PEP 484 introduced **Type Hints** (type annotations) to enable static analysis. While type hints have no effect on Python's runtime execution behavior, tools like `mypy`, Pyright, and modern IDEs use them to detect bugs, enforce contracts, and enhance autocomplete capabilities.

---

## 1. Basic Type Annotations

You can annotate variables, function parameters, and function return values.

### Variable Annotations (PEP 526)
Variables are annotated using the syntax `variable_name: type = value`.

### Generics in Collections (PEP 585)
Starting in Python 3.9, standard collection classes (`list`, `dict`, `set`, `tuple`) can be used directly as generic types, replacing the legacy capital-letter types imported from the `typing` module (e.g. `list[int]` instead of `typing.List[int]`).

In [ ]:
# Variable type hints
item_count: int = 42
price_per_item: float = 1.99
store_name: str = "Pythonic Goods"
is_open: bool = True

# Generic collection type hints
shopping_list: list[str] = ["apples", "bananas", "oranges"]
inventory: dict[str, int] = {"apples": 10, "bananas": 5}

# Function type annotations
def apply_discount(original_price: float, discount: float) -> float:
    return original_price * (1.0 - discount)

discounted: float = apply_discount(100.0, 0.15)
print("Discounted Price:", discounted)

---

## 2. Union and Optional Types (PEP 604)

Often, a variable or function argument can accept multiple different types.
- **Union Types:** Denotes that a value can be any of the specified types. In Python 3.10+, you can use the pipe operator (`|`) to write unions (e.g. `int | float`), replacing `typing.Union[int, float]`.
- **Optional Types:** Denotes that a value is either of a specified type or `None`. In modern Python, this is represented as `Type | None`.

In [ ]:
def fetch_port(config: dict[str, int | str]) -> int | None:
    # Returns an integer port, parses a string port, or returns None if missing
    port = config.get("port")
    if port is None:
        return None
    if isinstance(port, str):
        return int(port)
    return port

print("Port (int):", fetch_port({"port": 8080}))
print("Port (str):", fetch_port({"port": "9000"}))
print("Port (None):", fetch_port({}))

## 3. Generics and Parametric Polymorphism

Generics allow you to write reusable algorithms and structures that enforce type consistency across different types.
- **`TypeVar`**: Declares a generic type variable. This variable binds to whatever concrete type is supplied during type-checking. You can constrain the allowed types (e.g., `T = TypeVar('T', int, float)`) or bound them (e.g., `T = TypeVar('T', bound=SomeBaseClass)`).
- **`Generic[T]`**: Subclassed to define a class that can be parameterized with type parameter `T`. By inheriting from `Generic[T]`, you specify that the class accepts parameter `T` when instantiated (e.g. `stack = Stack[int]()`), allowing static type checkers to verify operations.
- **Type Invariance, Covariance, and Contravariance**: By default, generic type parameters are invariant. If you need a container to be covariant (e.g., if class `Dog` is a subclass of `Animal`, then `Container[Dog]` is a subclass of `Container[Animal]`), you define `TypeVar('T', covariant=True)`.

In [ ]:
from typing import TypeVar, Generic

T = TypeVar('T')  # Declare type variable

class Container(Generic[T]):
    def __init__(self, value: T):
        self._value: T = value

    def get(self) -> T:
        return self._value

    def set(self, value: T) -> None:
        self._value = value

# Constructing typed instances
int_container = Container[int](10)
str_container = Container[str]("Python")

print("int container content:", int_container.get())
print("str container content:", str_container.get())

## 4. Structural Subtyping: Protocols (PEP 544)

Python is structurally typed (duck-typed) at its core. However, standard object inheritance represents nominal typing (subtyping based on class name declarations).

PEP 544 introduced **Protocols** (`typing.Protocol`) to support static structural subtyping.
- **Static Duck-Typing**: If a class implements the exact attributes and methods declared in a Protocol class, a static checker like `mypy` or Pyright will treat it as a subtype of that Protocol, even if it does not inherit from it.
- **Defining a Protocol**: You define a protocol by subclassing `typing.Protocol` and defining the required methods and attributes (using `...` as the body).
- **Runtime Checkable Protocols**: Protocols can be decorated with `@typing.runtime_checkable` to allow their use with standard `isinstance()` and `issubclass()` checks at runtime. This inspects the object for the required methods rather than its class inheritance tree.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Flyable(Protocol):
    def fly(self) -> str:
        ...

class Airplane:
    def fly(self) -> str:
        return "Engines roaring!"

class Bird:
    def fly(self) -> str:
        return "Flapping wings!"

def launch(subject: Flyable) -> None:
    print("Launching:", subject.fly())

launch(Airplane())
launch(Bird())

# Runtime checkable verification using isinstance/issubclass
print("\nRuntime checks:")
print("Is Airplane instance Flyable?", isinstance(Airplane(), Flyable))
print("Is Bird class a subclass of Flyable?", issubclass(Bird, Flyable))


## 5. Special Type Qualifiers

- **`Any`**: A dynamic escape hatch. Disables type checking for that value; any operation is permitted.
- **`Callable[[Arg1, Arg2], Return]`**: Represents callable signatures (functions, lambdas, or objects implementing `__call__`). The first element is a list of parameter types, and the second element is the return type. For example, `Callable[[int, str], bool]` indicates a function that accepts an `int` and a `str` and returns a `bool`. If the parameter list is not specified or arbitrary, you can write `Callable[..., Return]`.
- **`Literal[...]`**: Restricts values to a specific set of exact literal values (e.g. `Literal['r', 'w', 'a']`).
- **`TypeAlias`**: Defines an alias name for complex compound type annotations.

In [ ]:
from typing import Callable, Literal, TypeAlias

# Custom Type Alias
Matrix: TypeAlias = list[list[float]]

def scale_matrix(matrix: Matrix, scale_fn: Callable[[float], float]) -> Matrix:
    return [[scale_fn(val) for val in row] for row in matrix]

def open_connection(mode: Literal['read', 'write']) -> None:
    print(f"Opening connection in {mode} mode.")

mat: Matrix = [[1.0, 2.0], [3.0, 4.0]]
doubled = scale_matrix(mat, lambda x: x * 2.0)
print("Scaled matrix:", doubled)

open_connection('read')
# open_connection('delete') # Static analysis warning here

---

## Coding Challenges

Complete the exercises below to apply static typing annotations to Python code.

### Challenge 1: Typing a Dynamic Python Function

Write a type-annotated function `parse_record` that takes:
1. `record`: A dictionary with string keys and any value types (`dict[str, Any]`).
2. `default_id` (Optional, defaults to `None`): An integer or `None` (`int | None`).

Specifications:
- Returns a tuple containing: `(id: int, name: str, email: str | None)`.
- If key `"id"` exists and is a valid integer (excluding booleans), use it. Otherwise, if `default_id` is a valid integer, use `default_id`. If both are missing or invalid, raise a `ValueError`.
- If key `"name"` is missing or is not a string, raise a `TypeError`.
- If key `"email"` is present but is not a string or `None`, raise a `TypeError`.
- Annotate the signature and body using modern type hint syntax.

In [ ]:
from typing import Any

def parse_record(record: dict[str, Any], default_id: int | None = None) -> tuple[int, str, str | None]:
    # TODO: Implement the parsing logic with correct validation and type annotations
    if "id" in record and isinstance(record["id"], int) and not isinstance(record["id"], bool):
        rec_id = record["id"]
    elif default_id is not None:
        rec_id = default_id
    else: 
        raise ValueError("Invalid or missing ID")

    if "name" not in record or not isinstance(record["name"], str):
        raise TypeError("Invalid or missing name")
    rec_name = record["name"]

    rec_email = record.get("email", None)
    if rec_email is not None and not isinstance(rec_email, str):
        raise TypeError("Invalid email type")

    return (rec_id, rec_name, rec_email)

In [ ]:
# Test Challenge 1
res1 = parse_record({"id": 10, "name": "Alice", "email": "alice@example.com"})
assert res1 == (10, "Alice", "alice@example.com")

res2 = parse_record({"name": "Bob"}, default_id=100)
assert res2 == (100, "Bob", None)

try:
    parse_record({"name": "Charlie"})
    assert False, "Allowed missing ID"
except ValueError:
    pass

try:
    parse_record({"id": "ten", "name": "Charlie"})
    assert False, "Allowed string ID"
except ValueError:
    pass

try:
    parse_record({"id": 5, "email": "charlie@example.com"})
    assert False, "Allowed missing name"
except TypeError:
    pass

try:
    parse_record({"id": 5, "name": "Charlie", "email": True})
    assert False, "Allowed non-string email"
except TypeError:
    pass

print("Challenge 1 passed!")

### Challenge 2: A Generic Stack Class

Implement a generic `Stack[T]` class representing a LIFO collection:
1. Subclass `Generic[T]` using a `TypeVar` name `T`.
2. Explicitly type internal list storage `self._items` to store instances of `T`.
3. Implement methods with complete annotations:
   - `push(self, item: T) -> None`
   - `pop(self) -> T` (raise `IndexError` if stack is empty)
   - `peek(self) -> T` (raise `IndexError` if stack is empty)
   - `is_empty(self) -> bool`
   - `__len__(self) -> int`

In [ ]:
from typing import TypeVar, Generic

T = TypeVar('T')

class Stack(Generic[T]):
    def __init__(self) -> None:
        # TODO: Initialize internal list with type hints
        self._items: list[T] = []

    def push(self, item: T) -> None:
        # TODO: Add item to stack
        self._items.append(item)

    def pop(self) -> T:
        # TODO: Pop item off stack
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self) -> T:
        # TODO: Peek top item of stack
        if self.is_empty():
            raise IndexError("peek from empty stack")
        return self._items[-1]

    def is_empty(self) -> bool:
        # TODO: Check if stack is empty
        return len(self._items) == 0

    def __len__(self) -> int:
        # TODO: Return element count
        return len(self._items)

In [ ]:
# Test Challenge 2
int_stack = Stack[int]()
assert int_stack.is_empty() is True
assert len(int_stack) == 0

int_stack.push(100)
int_stack.push(200)
assert int_stack.is_empty() is False
assert len(int_stack) == 2
assert int_stack.peek() == 200
assert int_stack.pop() == 200
assert int_stack.pop() == 100

try:
    int_stack.pop()
    assert False, "Popped empty stack"
except IndexError:
    pass

# String validation
str_stack = Stack[str]()
str_stack.push("hello")
assert str_stack.peek() == "hello"
assert isinstance(str_stack.pop(), str)

print("Challenge 2 passed!")

### Challenge 3: Duck-Typing Contract Validation using Protocol

Demonstrate structural subtyping in Python using `Protocol`:
1. Declare a class `Serializable(Protocol)` requiring a single method signature: `serialize(self) -> str`.
2. Write a consumer function `save_to_file(obj: Serializable, filename: str) -> None` that serializes the object using `obj.serialize()` and writes the output string to the designated filename.
3. Create a class `User` (with fields/properties `user_id: int` and `username: str`) that implements `serialize(self) -> str` returning a JSON representation of itself (e.g. `"{\"user_id\": 123, \"username\": \"alice\"}"`). 
4. Note: `User` must **not** inherit from `Serializable` directly.

In [ ]:
from typing import Protocol
import json

class Serializable(Protocol):
    def serialize(self) -> str:
        ...

def save_to_file(obj: Serializable, filename: str) -> None:
    # TODO: Serialize the object and write it to the specified file
    content = obj.serialize()
    with open(filename, 'w') as f:
        f.write(content)

class User:
    def __init__(self, user_id: int, username: str):
        self.user_id = user_id
        self.username = username

    def serialize(self) -> str:
        # TODO: Return user as a serialized JSON string
        return json.dumps({"user_id": self.user_id, "username": self.username})

In [ ]:
# Test Challenge 3
import os

test_user = User(456, "dustin")
filename = "user_serialize_test.json"

try:
    # Check if save_to_file successfully uses the User object (conforming structurally to Serializable)
    save_to_file(test_user, filename)
    assert os.path.exists(filename) == True

    with open(filename, "r") as f:
        parsed = json.load(f)
    
    assert parsed["user_id"] == 456
    assert parsed["username"] == "dustin"
finally:
    if os.path.exists(filename):
        os.remove(filename)

print("Challenge 3 passed!")

[◀ Notebook 10: Magic Methods](10_advanced_data_model.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 12: Iterators & Generators ▶](12_iterators_and_generators.ipynb)